# Equipment, Connector, and Port Scenarios

## Goal

Create a planning-level shortlist of Level 2 and DC-fast sites, estimate initial and expansion port counts under three EV-adoption scenarios, and recommend power bands and connector mixes.

> Port counts are transparent scenario estimates, not forecasts or engineered designs. NREL EVI-Pro/EVI-X, utility load studies, travel-demand modeling, and site-specific utilization data should replace these screening formulas before investment.

## Context & Assumptions

- NREL's national 2030 analysis estimated approximately 1 million public Level 2 ports for roughly 33 million light-duty EVs, or about one public Level 2 port per 33 EVs.
- Level 2 need is adjusted upward where renter and multifamily shares indicate greater dependence on public charging.
- DC-fast need uses nearby AADT, an assumed share of traffic that is electric, an assumed 1.5% daily fast-charging stop rate, and 14 useful sessions per port per day.
- Sites are reduced to one mapped opportunity per block group to limit obvious double counting.
- J1772 and J3400 are recommended for Level 2; CCS and J3400 are recommended for new DC-fast sites. CHAdeMO is retained only as a legacy-market review item.

In [ ]:
from pathlib import Path
import math

import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability")
SPATIAL_DIR = PROJECT_ROOT / "outputs" / "spatial"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
for folder in [SPATIAL_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

screening_path = SPATIAL_DIR / "level2_candidate_environmental_screening.gpkg"
demand_path = SPATIAL_DIR / "acs_ev_demand_block_groups.gpkg"
corridor_path = SPATIAL_DIR / "dc_fast_corridor_demand.gpkg"
for path in [screening_path, demand_path, corridor_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required prior output is missing: {path.name}")

candidates = gpd.read_file(screening_path, layer="screened_candidates")
demand = gpd.read_file(demand_path, layer="ev_demand")
corridor = gpd.read_file(corridor_path, layer="corridor_demand")

SCENARIOS = {
    "current": {"ev_share": 0.04},
    "moderate": {"ev_share": 0.10},
    "high": {"ev_share": 0.20},
}
NREL_PUBLIC_L2_PORTS_PER_EV = 1_000_000 / 33_000_000
DC_FAST_STOP_RATE = 0.015
DC_SESSIONS_PER_PORT_DAY = 14

def round_up_even(value, minimum=4, maximum=24):
    bounded = min(maximum, max(minimum, int(math.ceil(value))))
    return bounded if bounded % 2 == 0 else min(maximum, bounded + 1)

## Data

### 1. Assemble eligible candidates with demand and corridor attributes

In [ ]:
eligible = candidates.loc[candidates["environmentally_eligible"].astype(bool)].copy()
demand_fields = [
    "GEOID", "households", "vehicle_households", "zero_vehicle_households",
    "renter_share", "multifamily_share", "market_demand_index", "public_access_need_index",
]
corridor_fields = [
    "GEOID", "aadt_value", "dc_corridor_score", "dc_corridor_priority", "nearest_dc_fast_miles",
]
sites = eligible.merge(
    demand[demand_fields], on="GEOID", validate="many_to_one"
).merge(
    corridor[corridor_fields], on="GEOID", validate="many_to_one"
)

sites["role"] = np.where(
    sites["dc_corridor_priority"] == "High relative priority", "DC fast", "Level 2"
)
sites["role_score"] = np.where(
    sites["role"] == "DC fast", sites["dc_corridor_score"], sites["level2_suitability_score"]
)

# Keep the highest-ranked mapped opportunity in each block group.
sites = sites.sort_values(["role_score", "screened_site_score"], ascending=False).drop_duplicates("GEOID")
dc_sites = sites.loc[sites["role"] == "DC fast"].head(12)
level2_sites = sites.loc[sites["role"] == "Level 2"].head(20)
selected = gpd.GeoDataFrame(pd.concat([dc_sites, level2_sites], ignore_index=True), crs=sites.crs)
selected["site_id"] = [f"EV-{index:03d}" for index in range(1, len(selected) + 1)]

assert not selected.empty
assert selected["GEOID"].is_unique
print("Selected DC-fast screening sites:", len(dc_sites))
print("Selected Level 2 screening sites:", len(level2_sites))

## Results

### 2. Estimate scenario port counts

In [ ]:
vehicle_households = (selected["vehicle_households"] - selected["zero_vehicle_households"]).clip(lower=0)
selected["public_charging_dependency"] = (
    0.20 + 0.35 * selected["renter_share"].fillna(0) + 0.25 * selected["multifamily_share"].fillna(0)
).clip(lower=0.20, upper=0.75)
selected["l2_access_adjustment"] = (selected["public_charging_dependency"] / 0.25).clip(0.75, 2.0)

scenario_rows = []
for row in selected.itertuples():
    for scenario, assumptions in SCENARIOS.items():
        ev_share = assumptions["ev_share"]
        projected_evs = vehicle_households.loc[row.Index] * ev_share
        if row.role == "Level 2":
            raw_ports = projected_evs * NREL_PUBLIC_L2_PORTS_PER_EV * row.l2_access_adjustment
            ports = round_up_even(raw_ports, minimum=4, maximum=24)
        else:
            estimated_sessions = row.aadt_value * ev_share * DC_FAST_STOP_RATE
            raw_ports = estimated_sessions / DC_SESSIONS_PER_PORT_DAY
            ports = round_up_even(raw_ports, minimum=4, maximum=24)
        scenario_rows.append({
            "site_id": row.site_id, "scenario": scenario, "ev_share": ev_share,
            "projected_local_evs": projected_evs, "raw_port_estimate": raw_ports,
            "recommended_ports": ports,
        })

scenario_table = pd.DataFrame(scenario_rows)
scenario_table.to_csv(TABLE_DIR / "ev_port_demand_scenarios.csv", index=False)
port_pivot = scenario_table.pivot(index="site_id", columns="scenario", values="recommended_ports")
port_pivot = port_pivot.rename(columns={
    "current": "current_ports", "moderate": "initial_ports", "high": "expansion_ports"
})
selected = selected.merge(port_pivot, on="site_id", validate="one_to_one")

### 3. Recommend charging power and connector mix

In [ ]:
def equipment_recommendation(row):
    if row["role"] == "Level 2":
        return pd.Series({
            "power_recommendation": "11.5 kW Level 2; evaluate up to 19.2 kW",
            "connector_recommendation": "J1772 and J3400",
            "legacy_connector_note": "None",
        })
    if row["dc_corridor_score"] >= 90:
        power = "250–350 kW DC fast"
    elif row["dc_corridor_score"] >= 75:
        power = "150–249 kW DC fast"
    else:
        power = "50–149 kW DC fast"
    return pd.Series({
        "power_recommendation": power,
        "connector_recommendation": "CCS and J3400",
        "legacy_connector_note": "Assess local CHAdeMO fleet; no default new CHAdeMO port",
    })

selected = pd.concat([selected, selected.apply(equipment_recommendation, axis=1)], axis=1)
selected["initial_connector_split"] = np.where(
    selected["role"] == "Level 2", "50% J1772 / 50% J3400", "50% CCS / 50% J3400"
)
selected["estimated_site_kw_initial"] = np.where(
    selected["role"] == "Level 2",
    selected["initial_ports"] * 11.5,
    selected["initial_ports"] * np.select(
        [selected["dc_corridor_score"] >= 90, selected["dc_corridor_score"] >= 75],
        [300, 180], default=100,
    ),
)

selected.to_file(SPATIAL_DIR / "recommended_ev_sites_and_equipment.gpkg", layer="recommendations", driver="GPKG")
recommendation_fields = [
    "site_id", "candidate_id", "name", "county_name", "role",
    "power_recommendation", "connector_recommendation", "initial_connector_split",
    "current_ports", "initial_ports", "expansion_ports", "estimated_site_kw_initial",
    "public_charging_dependency", "level2_suitability_score", "dc_corridor_score",
    "protected_land_review", "legacy_connector_note", "osm_type", "osm_id",
]
recommendations = selected[recommendation_fields].sort_values(["role", "initial_ports"], ascending=[True, False])
recommendations.to_csv(TABLE_DIR / "recommended_ev_sites_equipment_and_ports.csv", index=False)
display(recommendations.round(2))

### 4. Summarize infrastructure by scenario and charging role

In [ ]:
scenario_with_role = scenario_table.merge(selected[["site_id", "role"]], on="site_id", validate="many_to_one")
network_summary = (
    scenario_with_role.groupby(["scenario", "role"], as_index=False)
    .agg(sites=("site_id", "nunique"), ports=("recommended_ports", "sum"))
)
network_summary.to_csv(TABLE_DIR / "recommended_network_ports_by_scenario.csv", index=False)
display(network_summary)

## Validation

In [ ]:
assert selected["site_id"].is_unique
assert selected["GEOID"].is_unique
assert len(scenario_table) == len(selected) * len(SCENARIOS)
assert scenario_table["recommended_ports"].between(4, 24).all()
assert (scenario_table["recommended_ports"] % 2 == 0).all()
assert (selected["current_ports"] <= selected["initial_ports"]).all()
assert (selected["initial_ports"] <= selected["expansion_ports"]).all()
assert selected["estimated_site_kw_initial"].gt(0).all()

expected_outputs = [
    SPATIAL_DIR / "recommended_ev_sites_and_equipment.gpkg",
    TABLE_DIR / "ev_port_demand_scenarios.csv",
    TABLE_DIR / "recommended_ev_sites_equipment_and_ports.csv",
    TABLE_DIR / "recommended_network_ports_by_scenario.csv",
]
assert all(path.exists() for path in expected_outputs)

print("All site-grain, scenario, monotonicity, port-bound, and output checks passed.")
print("Validation status: Planning scenario only; calibrate with EVI-Pro and utility/site studies.")

## Next Steps

Create the final recommendation map and portfolio-ready tables, then document methodology, limitations, sources, and future improvements in the project README.